Data about unemployment on municipality level are loaded from https://www.regionalstatistik.de/genesis//online?operation=table&code=13211-01-03-5&bypass=true&levelindex=1&levelid=1762338172400#abreadcrumb

Data contain ID column as identifier.

ID consist of:
- Land (2 digits) which represents Bundesland
- Regierungsbezirk (1 digit) which represents governmental district
- Kreis (2 digits) which represents district
- Gemeinde (3 digits) which represents municipality

There are also rows representing higher administrative levels (e.g. districts) which can be identified by having just two or five digits in ID column.
We cannot just filter them out because Hamburg has no districts and its municipalities have just two digits in ID column.

In [ ]:
# load municipalities for filtering and merging
from geoscore_de.data_flow.features.municipality import MunicipalityFeature

municipalities = MunicipalityFeature("../../data/raw/municipalities_2022.csv")
municipalities = municipalities.load_transform()
municipalities.head()

# Load of unemployment data

In [ ]:
from geoscore_de.data_flow.features.unemployment import load_unemployment_data

df = load_unemployment_data("../../data/raw/features/13211-01-03-5-unemployment-2025.csv")

In [ ]:
df.dtypes

In [ ]:
df_merged = municipalities.merge(df, on="AGS", how="left")
df_merged.head()

In [ ]:
df_merged["unemployment_rate"] = df_merged["unemployment_total"] / df_merged["Persons"]

In [ ]:
df_merged

In [ ]:
import plotnine as gg

# show histogram of unemployment rate
(
    gg.ggplot(df_merged, gg.aes(x="unemployment_rate"))
    + gg.geom_histogram(binwidth=0.001)
    + gg.scale_x_continuous(labels=lambda labels: ["{:.0f}%".format(x * 100) for x in labels])
    + gg.labs(
        title="Distribution of Unemployment Rate in German Municipalities",
        x="Unemployment Rate",
        y="Count of Municipalities",
    )
)

In [ ]:
# now show it on map of germany
from geoscore_de.data_flow.geo import load_geo_data

gdf = load_geo_data("../../data/georef-germany-gemeinde.csv")

In [ ]:
# join data by ags
gdf_merged = gdf.merge(df_merged, on="AGS", how="outer", indicator=True)

In [ ]:
# print count of missing values
gdf_merged["_merge"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

gdf_merged.plot(
    column="unemployment_rate",
    legend=True,
    cmap="coolwarm",
    figsize=(10, 10),
    vmin=0,
    vmax=0.05,
    missing_kwds={"color": "black"},
)
plt.title("Unemployment Rate in German Municipalities")
plt.show()

## Compare year 2025 with 2020
Now we will compare unemployment data from 2025 with data from 2020. 
We will subtract 2020 data from 2025 data to get the change in unemployment.

In [ ]:
from geoscore_de.data_flow.features.unemployment import load_unemployment_data

df_2020 = load_unemployment_data("../../data/raw/features/13211-01-03-5-unemployment-2020.csv")
df_2020

In [ ]:
mmerged = gdf_merged.merge(df_2020, on="AGS", suffixes=("_2025", "_2020"), how="left")
mmerged["unemployment_rate_2020"] = mmerged["unemployment_total_2020"] / mmerged["Persons"]

mmerged

In [ ]:
mmerged["yty_unemployment_rate"] = mmerged["unemployment_rate"] - mmerged["unemployment_rate_2020"]
mmerged[["AGS", "unemployment_rate", "unemployment_rate_2020", "yty_unemployment_rate"]]

In [ ]:
import plotnine as gg

# show histogram of unemployment rate
(
    gg.ggplot(mmerged, gg.aes(x="yty_unemployment_rate"))
    + gg.geom_histogram(binwidth=0.001)
    + gg.scale_x_continuous(labels=lambda labels: ["{:.0f}%".format(x * 100) for x in labels])
    + gg.labs(
        title="Change in Unemployment Rate from 2020 to 2025 in German Municipalities",
        x="Unemployment Rate",
        y="Count of Municipalities",
    )
)

In [ ]:
import matplotlib.pyplot as plt

mmerged.plot(
    column="yty_unemployment_rate",
    legend=True,
    cmap="coolwarm",
    figsize=(10, 10),
    vmin=-0.03,
    vmax=0.03,
    missing_kwds={"color": "black"},
)
plt.title("Change in Unemployment Rate from 2020 to 2025 in German Municipalities")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch


def classify_change(x):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return np.nan
    return 1 if x > 0 else 0


mmerged["yty_unemployment_rate_id"] = mmerged["yty_unemployment_rate"].apply(lambda x: classify_change(x))

ax = mmerged.plot(
    column="yty_unemployment_rate_id",
    categorical=True,
    legend=False,
    cmap="coolwarm",
    figsize=(10, 10),
    missing_kwds={"color": "black"},
)


legend_handles = [
    Patch(facecolor=plt.cm.coolwarm(1.0), edgecolor="white", label="1 = increase"),
    Patch(facecolor=plt.cm.coolwarm(0.0), edgecolor="white", label="0 = decrease"),
]

ax.legend(handles=legend_handles, title="Change in unemployment rate")
plt.title("Change in Unemployment Rate from 2020 to 2025 identifier")
plt.show()